# Step 1 - Transform the OBS data into a decay time series

This notebook is designed specifically for processing a decaying OBS time series where a tank was in a state of mixing and then the mixer was turned off. The time following the mixer being turned off is referred to as the decay since concentration decreases as material settles.

In [ ]:
# --- Find the project folder, then work inside its work/ folder ----------------
# All data paths below (data_OBS/, calibrations/, results/, ...) are relative to work/.
import os
import sys
_root = os.getcwd()
for _ in range(4):   # this folder, then up to three levels above
    if os.path.isdir(os.path.join(_root, 'tools')) and os.path.isdir(os.path.join(_root, 'work_template')):
        break
    _root = os.path.dirname(_root)
else:
    raise FileNotFoundError('could not find the project folder (the one containing tools/ and work_template/)')
sys.path.insert(0, _root)                          # for "from tools ..." imports
sys.path.insert(0, os.path.join(_root, 'tools'))   # for "import settling_tools"
if not os.path.isdir(os.path.join(_root, 'work')):
    from setup_work import setup_work
    setup_work(_root)                              # first run: create work/ from work_template/
os.chdir(os.path.join(_root, 'work'))
print('working in:', os.getcwd())
# ---------------------------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%config InlineBackend.figure_format='retina'

%matplotlib widget

import glob
from datetime import datetime, timedelta

## Find the OBS files

Note that this works with `.csv` files. You might need to change the extension of the natural OBS output file.

In [ ]:
##  --- User input ----------------------------- 

dir_working = os.getcwd() # working directory, change to absolute path if running remote
dir_data = 'data_OBS/' # relative path to the data with respect to the working directory
ext = '*.csv'     # data file extension
##   ------------------------------------------- 


# Find all of the data files and store the names 

files = sorted(glob.glob(dir_data+ext))
nf=len(files)
os.chdir(dir_data)
filenames = sorted(glob.glob(ext))
for i in range(0,nf):
    filenames[i]=os.path.splitext(os.path.basename(filenames[i]))[0] #strips extension
os.chdir(dir_working)
    
#Print the file names and index value for each file

for i in range(0,nf):
    print('file: ',filenames[i],'(index: ',i,')')
 

Select the OBS file you want to work with

In [ ]:
# --- User input ----------------------------- 
# select the file to look at by name; or set j to an index from the list above (e.g. j = 0)
j = filenames.index('MicroSphere_75_90_SAMPLE')
#  ------------------------------------------- 

print('file: ',filenames[j],'(index: ',j,')')

## Read in the OBS data

In [ ]:
data = pd.read_csv(files[j], header=None,skiprows=4)

# --------------------------------------------
# for some reason, the third column (index 2) should be the last... so swap here. Comment out the three lines below if you don't want to do the swap

cols = data.columns.tolist() # Get the list of all column indices
cols.append(cols.pop(2)) # Remove column 2 and append it to the end
data = data[cols] # Reorder the DataFrame columns
# --------------------------------------------

data

## Time corrections

Enter any time corrections to the data that are needed

In [ ]:
# ------- time correction ----------------------
hr_cor = 0 # hours to add/subtract
min_cor = -1 # minutes to add/subtract 
             # [the OBS logger time associated with the row is the time of the last column in the row]
sec_cor = 2 # seconds to add/subtract
# ---------------------------------------------

start_time = datetime.strptime(data[0][0],'%Y-%m-%d %H:%M:%S')
start_time = start_time + timedelta(hours=hr_cor,minutes=min_cor,seconds=sec_cor)
start_time_value = start_time.strftime("%Y%m%d%H%M%S")

print('start time: ',start_time)
print('start time value: ',start_time_value)


Plot the reformatted data

In [ ]:
# reform the dataframe to provide a time series of NTU values

NTU_by_min = data.drop([0,1], axis=1)            # drop the first two columns
samples_per_min = len(NTU_by_min.columns)        # find the number of samples per minute
scan_int_sec = 60 / samples_per_min              # find the scan interval in seconds
new_columns = list(range(0, samples_per_min))    # renumber the columns
NTU_by_min.columns = new_columns

transposed_data = NTU_by_min.transpose()        # transpose the dataframe
NTU_by_sec = pd.melt(transposed_data)           # stack all the columns into a single column
NTU_by_sec.columns = ['min_running', 'NTU']     # rename the columns
NTU_by_sec.insert(0, 'second_running', NTU_by_sec.index*scan_int_sec+scan_int_sec) # put the running time in seconds in the first column [added the scan interval so that the last value is aligned with the minute]

running_time = start_time + pd.to_timedelta(NTU_by_sec['second_running'], unit='s') # use datetime
NTU_by_sec.insert(0, 'DateTime', running_time)
NTU_by_sec.insert(1, 'DateTime_values', running_time.dt.strftime("%Y%m%d%H%M%S"))

display(NTU_by_sec.head())

fig, ax = plt.subplots()
ax.plot(NTU_by_sec['second_running'], NTU_by_sec['NTU'], ls='-')
# ax.plot(NTU_by_sec['DateTime'], NTU_by_sec['NTU'], ls='-')           
ax.set_xlabel(r'Time [s]')
ax.set_ylabel(r'Turbidity [NTU]');

# save the dataframe to a CSV
NTU_by_sec.to_csv(dir_data+filenames[j]+'_time_series.csv', index=False)

## Extra processing for decay analysis

Use the plot above to identify the start and end times for defining the initial and background concentrations. Data that sits between these times will be averaged and used as the initial and background concentration in future analysis. 

The user also needs to specify when the mixer is turned off. This can be done by specifying when the mixer is turned on (as identified in the OBS time series) and then adding to that the time between when the mixer was turned on and when it was turned off. For example, `mixer_on = 110` and then `mixer_off = mixer_on+4*60`. One can also just input the time in seconds when the mixer was turned off, e.g. `mixer_off = 350`. The time when the mixer is turned off is the critical value. The time for the mixer being turned on is only helpful or needed if the exact time when the mixer was turned off is not known outside of its timing relative to the mixer on time.

In [ ]:
# - user input --------------------------------
Co_start_s = 120  # start time for initial concentration averaging window
Co_end_s = 340    # end time for initial concentration averaging window

Cbackground_start_s = 700 # start time for background concentration averaging window
Cbackground_end_s = 1014 # end time for background concentration averaging window

mixer_on = 110 # time when mixer was turned on (not used outside of defining when the mixer turns off)
mixer_off = mixer_on+4*60
# mixer_off = 350 # time in seconds when the mixer was turned off

# --------------------------------------------

# calculate the averages
Co_NTU_avg = NTU_by_sec.loc[NTU_by_sec['second_running'].between(Co_start_s, Co_end_s),'NTU'].mean()
Cbackground_NTU_avg = NTU_by_sec.loc[NTU_by_sec['second_running'].between(Cbackground_start_s, Cbackground_end_s),'NTU'].mean()

# plot the values as a check
fig, ax = plt.subplots()
ax.plot(NTU_by_sec['second_running'], NTU_by_sec['NTU'], ls='-')
ax.plot([Co_start_s,Co_end_s], [Co_NTU_avg,Co_NTU_avg], color='C3',ls='--', label='$C_o$')
ax.plot([Cbackground_start_s,Cbackground_end_s], [Cbackground_NTU_avg,Cbackground_NTU_avg], color='C2',ls=':', label='$C_{background}$')
ax.set_xlabel(r'Time [s]')
ax.set_ylabel(r'Turbidity [NTU]')

ax.axvline(mixer_on, color='gray', linestyle='--', label='mixer on')
ax.axvline(mixer_off, color='black', linestyle='--', label='mixer off')
ax.legend();


## Trim the time series

No user input is needed. The cell below takes input that was given in the cells above and generates a concentration decay-only time series. This is the time series that will be used in future analysis.

In [ ]:
OBS_decay = NTU_by_sec.loc[
    NTU_by_sec['second_running'] > mixer_off
].copy()

# plot the values as a check
fig, ax = plt.subplots()
ax.plot(OBS_decay['second_running'], OBS_decay['NTU'], ls='-')
ax.axhline(y=Co_NTU_avg, color='C3', linestyle='--', label='$C_o$')
ax.axhline(y=Cbackground_NTU_avg, color='C2',ls=':', label='$C_{background}$')
ax.set_xlabel(r'Time [s]')
ax.set_ylabel(r'Turbidity [NTU]')
ax.legend();

# add a column that is seconds since the mixer was turned off
OBS_decay['sec'] = OBS_decay['second_running']-mixer_off

# store the average start and end NTU values for further analysis
OBS_decay['Co_NTU_avg'] = Co_NTU_avg
OBS_decay['Cbackground_NTU_avg'] = Cbackground_NTU_avg

OBS_decay.to_csv(dir_data+filenames[j]+'_time_series_decayONLY.csv', index=False)

## Change to mass/vol concentration

The cells below are optional. They are not needed for settling analysis. They can be helpful in contextualizing the results. However, note that the calibration coefficients $m$ and $b$ (defined below) are needed; the calibration itself is done outside this notebook.

The assumption is that the OBS NTU values can be converted to mass concentration using

$$
C_{mass} = m C_{NTU} + b
$$

where $C_{mass}$ is the mass concentration in mg/L, $C_{NTU}$ is the concentration in units of NTU and $m$ and $b$ are the slope and intercept of the empirical calibration equation between the two

In [ ]:
# --- User input ----------------------------- 
# Single-size particle standards calibration:
#     Concentration (mg/L) = 54.1703 * NTU - 87.3999   (R2 = 0.9835, n = 12)
#
# Fine-sand calibration:
#     Concentration (mg/L) = 39.07848275 * NTU - 78.31310717
#
# Coarse-sand calibration:
#     Concentration (mg/L) = 72.38610837 * NTU - 153

m = 54.1703  # enter the slope coefficient
b = - 87.3999 # enter the intercept

# -----------------------------------------------------

# make the conversion
NTU_by_sec['C_mg_l'] = m*NTU_by_sec['NTU']+b

# save the dataframes with the mg/L to a CSV
NTU_by_sec.to_csv(dir_data+filenames[j]+'_time_series.csv', index=False)

# plot the full time series just to check
fig, ax = plt.subplots()
ax.plot(NTU_by_sec['second_running'], NTU_by_sec['C_mg_l'], ls='-')
# ax.plot(NTU_by_sec['DateTime'], NTU_by_sec['NTU'], ls='-')           
ax.set_xlabel(r'Time [s]')
ax.set_ylabel(r'$C$ [mg/L]');

# check to see if there is a decay time series, if so, update it
if isinstance(globals().get('OBS_decay'), pd.DataFrame) == True:
    OBS_decay['C_mg_l']=m*OBS_decay['NTU']+b
    OBS_decay['Co_mgL_avg'] = m*Co_NTU_avg+b
    OBS_decay['Cbackground_mgL_avg'] = m*Cbackground_NTU_avg+b
    OBS_decay.to_csv(dir_data+filenames[j]+'_time_series_decayONLY.csv', index=False)
